In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

WORK = Path("pd_beta_three")
METRICS = ["rate_hz", "dur_ms", "time_in_burst", "env_cv", "burst_amp", "beta_peak_db"]


def holm(p):
    p = np.asarray(p, float)
    out = np.full(p.shape, np.nan)
    ok = np.where(np.isfinite(p))[0]
    run = 0.0
    for i, j in enumerate(ok[np.argsort(p[ok])]):
        run = max(run, min(1.0, (len(ok) - i) * p[j]))
        out[j] = run
    return out


def ols(d, y, covs, term="is_pd"):
    """OLS of y on term + covs. Returns β, SE, p, n, standardised β and its 95 % CI (SD units of y)."""
    d = d[[y, term] + [c for c in covs if c != term]].dropna()
    xs = [term] + [c for c in covs if c != term]
    X = np.column_stack([np.ones(len(d))] + [d[c].astype(float).values for c in xs])
    yy = d[y].astype(float).values
    beta, *_ = np.linalg.lstsq(X, yy, rcond=None)
    r = yy - X @ beta
    dof = len(yy) - X.shape[1]
    se = np.sqrt(np.diag(r @ r / dof * np.linalg.pinv(X.T @ X)))
    sd = np.std(yy, ddof=1)
    tc = stats.t.ppf(0.975, dof)
    return {"beta": beta[1], "se": se[1], "p": 2 * stats.t.sf(abs(beta[1] / se[1]), dof), "n": len(d),
            "std_beta": beta[1] / sd, "ci_low": (beta[1] - tc * se[1]) / sd, "ci_high": (beta[1] + tc * se[1]) / sd}


cov = pd.read_csv(WORK / "preprocessed" / "iowa" / "covariates.csv").rename(columns={"out_name": "stem"})
qc = pd.read_csv(WORK / "preprocessed" / "iowa" / "preprocessing_qc.csv").rename(columns={"out_name": "stem"})
qc["clean_s"] = qc["rest_seconds"] * (1 - qc["rest_bad_fraction"])
extra = qc[["stem", "ic_muscle_artifact", "n_ics_removed", "n_bad_channels", "clean_s"]]

samples = {"gated (primary)": WORK / "results" / "iowa_full_per_recording.csv",
           "no gate": WORK / "results" / "iowa_full_per_recording_no_gate.csv"}
MODELS = {"M0": ["age", "male", "moca"],
          "M1": ["age", "male", "moca", "ic_muscle_artifact", "n_ics_removed"],
          "M2": ["age", "male", "moca", "ic_muscle_artifact", "n_ics_removed", "aperiodic_slope"],
          "M3": ["age", "male", "moca", "ic_muscle_artifact", "n_ics_removed", "aperiodic_slope", "clean_s",
                 "n_bad_channels"]}


def load(f):
    d = pd.read_csv(f).merge(cov[["stem", "age", "sex", "moca"]], on="stem", how="left").merge(extra, on="stem",
                                                                                                how="left")
    d["is_pd"] = (d.group == "PD").astype(float)
    d["male"] = d["sex"].astype(str).str.upper().str.startswith("M").astype(float).where(d["sex"].notna())
    return d


rows, balance = [], []
for name, f in samples.items():
    d = load(f)
    print(f"{name}: {int((d.is_pd == 1).sum())} PD, {int((d.is_pd == 0).sum())} HC")
    for c in ("ic_muscle_artifact", "n_ics_removed", "aperiodic_slope", "clean_s", "n_bad_channels"):
        a, b = d.loc[d.is_pd == 1, c].dropna(), d.loc[d.is_pd == 0, c].dropna()
        balance.append({"sample": name, "variable": c, "PD median": a.median(), "HC median": b.median(),
                        "mann_whitney_p": stats.mannwhitneyu(a, b).pvalue})
    for mname, covs in MODELS.items():
        t = pd.DataFrame([{**ols(d, m, covs), "metric": m} for m in METRICS])
        t["p_holm"] = holm(t.p)
        t["sample"], t["model"] = name, mname
        rows.append(t)

balance = pd.DataFrame(balance)
out = pd.concat(rows, ignore_index=True)
print("\nGroup balance of possible confounds:")
print(balance.round(3).to_string(index=False))
for name in samples:
    t = out[out["sample"] == name]
    print(f"\n{name}: PD − HC, standardised β [95 % CI] (Holm p) per model")
    piv = t.assign(cell=t.apply(lambda r: f"{r.std_beta:+.2f} [{r.ci_low:+.2f}, {r.ci_high:+.2f}] ({r.p_holm:.3f})",
                                axis=1)).pivot(index="metric", columns="model", values="cell").reindex(METRICS)
    print(piv.to_string())
out.to_csv(WORK / "results" / "iowa_confound_check.csv", index=False)
balance.to_csv(WORK / "results" / "iowa_confound_balance.csv", index=False)

gated (primary): 48 PD, 32 HC
no gate: 87 PD, 44 HC

Group balance of possible confounds:
         sample           variable  PD median  HC median  mann_whitney_p
gated (primary) ic_muscle_artifact      4.500      8.500           0.002
gated (primary)      n_ics_removed      8.000     11.500           0.006
gated (primary)    aperiodic_slope     -0.834     -0.875           0.566
gated (primary)            clean_s    138.515    152.900           0.093
gated (primary)     n_bad_channels      0.500      1.000           0.677
        no gate ic_muscle_artifact      3.000      7.500           0.000
        no gate      n_ics_removed      7.000     11.000           0.001
        no gate    aperiodic_slope     -0.621     -0.848           0.005
        no gate            clean_s    135.700    147.370           0.038
        no gate     n_bad_channels      0.000      1.000           0.084

gated (primary): PD − HC, standardised β [95 % CI] (Holm p) per model
model                               

In [2]:
d = load(samples["gated (primary)"])
cols = ["ic_muscle_artifact", "n_ics_removed", "aperiodic_slope", "burst_amp", "is_pd"]
print("Spearman correlations (gated sample):")
print(d[cols].corr(method="spearman").round(2).to_string())
r = ols(d, "aperiodic_slope", ["ic_muscle_artifact"])
print(f"\nPD − HC aperiodic slope with muscle ICs held constant: std β {r['std_beta']:+.2f} "
      f"[{r['ci_low']:+.2f}, {r['ci_high']:+.2f}], p {r['p']:.3f}, n {r['n']}")
r = ols(d, "burst_amp", MODELS["M2"] + ["is_pd"], term="aperiodic_slope")
print(f"Burst amplitude on aperiodic slope in M2 (group and other covariates held constant): "
      f"β {r['beta']:+.3f} per unit slope (SE {r['se']:.3f}), p {r['p']:.4f}")
print(f"\nSaved: {WORK / 'results' / 'iowa_confound_check.csv'}")

Spearman correlations (gated sample):
                    ic_muscle_artifact  n_ics_removed  aperiodic_slope  burst_amp  is_pd
ic_muscle_artifact                1.00           0.90             0.19       0.22  -0.35
n_ics_removed                     0.90           1.00             0.16       0.17  -0.31
aperiodic_slope                   0.19           0.16             1.00      -0.32   0.07
burst_amp                         0.22           0.17            -0.32       1.00  -0.36
is_pd                            -0.35          -0.31             0.07      -0.36   1.00

PD − HC aperiodic slope with muscle ICs held constant: std β +0.31 [-0.16, +0.78], p 0.195, n 80
Burst amplitude on aperiodic slope in M2 (group and other covariates held constant): β -0.865 per unit slope (SE 0.257), p 0.0012

Saved: pd_beta_three\results\iowa_confound_check.csv
